In [3]:
import sqlite3
import random
from datetime import datetime, timedelta
from faker import Faker
import pandas as pd

fake = Faker('es_AR')

def generar_dataset_simulado(num_clientes=150, num_compras=300):
    conn = sqlite3.connect('tienda_crochet.db')
    cursor = conn.cursor()

    # 1. Crear tablas para el historial transaccional
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS CLIENTES (
        id_cliente TEXT PRIMARY KEY,
        nombre TEXT NOT NULL,
        email TEXT NOT NULL,
        red_origen TEXT NOT NULL,
        nivel_experiencia TEXT NOT NULL
    );
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS TRANSACCIONES (
        id_transaccion TEXT PRIMARY KEY,
        id_cliente TEXT NOT NULL,
        fecha_hora TEXT NOT NULL,
        acepto_cross_selling INTEGER NOT NULL,
        FOREIGN KEY (id_cliente) REFERENCES CLIENTES(id_cliente)
    );
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS DETALLE_TRANSACCION (
        id_detalle INTEGER PRIMARY KEY AUTOINCREMENT,
        id_transaccion TEXT NOT NULL,
        tipo_producto TEXT NOT NULL,
        codigo_producto TEXT NOT NULL,
        cantidad INTEGER NOT NULL,
        FOREIGN KEY (id_transaccion) REFERENCES TRANSACCIONES(id_transaccion)
    );
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS INTERACCIONES_ASISTENTE (
        id_interaccion INTEGER PRIMARY KEY AUTOINCREMENT,
        id_cliente TEXT NOT NULL,
        patron_consultado TEXT NOT NULL,
        insumos_sugeridos TEXT NOT NULL,
        conversion_exitosa INTEGER NOT NULL,
        tiempo_sesion_segundos INTEGER NOT NULL,
        FOREIGN KEY (id_cliente) REFERENCES CLIENTES(id_cliente)
    );
    """)

    # 2. Generar Clientes
    redes = ['Instagram', 'TikTok', 'Pinterest', 'Facebook']
    niveles = ['Principiante', 'Intermedio', 'Avanzado']
    patrones_disponibles = ['GoPm', 'BuIn', 'PoAl', 'SnVe', 'SnIn']

    # Mapeo de receta para simular la compra cruzada coherente
    recetas_dict = {
        'GoPm': ('Hz', 'Al6'),
        'BuIn': ('Cs', 'Al8'),
        'PoAl': ('Tr-CS-SG', 'Al6'),
        'SnVe': ('Tr-HA-86', 'Mp4'),
        'SnIn': ('Tr-CS-SG', 'Ms6')
    }

    clientes = []
    for i in range(1, num_clientes + 1):
        cid = f"CLI-{i:03d}"
        clientes.append((
            cid,
            fake.name(),
            fake.email(),
            random.choice(redes),
            random.choices(niveles, weights=[0.5, 0.35, 0.15])[0]
        ))
    cursor.executemany("INSERT OR REPLACE INTO CLIENTES VALUES (?,?,?,?,?)", clientes)

    # 3. Generar Transacciones e Interacciones
    fecha_base = datetime(2026, 8, 1)

    for t in range(1, num_compras + 1):
        tid = f"TRX-{t:04d}"
        cliente_id = f"CLI-{random.randint(1, num_clientes):03d}"
        fecha = fecha_base + timedelta(days=random.randint(0, 60), minutes=random.randint(0, 1440))
        fecha_str = fecha.strftime("%Y-%m-%d %H:%M:%S")

        # El cliente compra al menos un patrón digital
        patron_elegido = random.choice(patrones_disponibles)

        # Probabilidad de aceptar venta cruzada recomendada por el asistente (40% de éxito)
        acepta_cross_selling = 1 if random.random() < 0.40 else 0

        cursor.execute("INSERT INTO TRANSACCIONES VALUES (?,?,?,?)",
                       (tid, cliente_id, fecha_str, acepta_cross_selling))

        # Detalle: patrón digital comprado
        cursor.execute("""
        INSERT INTO DETALLE_TRANSACCION (id_transaccion, tipo_producto, codigo_producto, cantidad)
        VALUES (?, 'PATRON', ?, 1)
        """, (tid, patron_elegido))

        # Si aceptó la recomendación, se agregan los insumos físicos exactos de la receta
        hilado_rec, aguja_rec = recetas_dict[patron_elegido]
        if acepta_cross_selling:
            cursor.execute("""
            INSERT INTO DETALLE_TRANSACCION (id_transaccion, tipo_producto, codigo_producto, cantidad)
            VALUES (?, 'HILADO', ?, 1)
            """, (tid, hilado_rec))

            cursor.execute("""
            INSERT INTO DETALLE_TRANSACCION (id_transaccion, tipo_producto, codigo_producto, cantidad)
            VALUES (?, 'AGUJA', ?, 1)
            """, (tid, aguja_rec))

        # Registrar interacción con el chatbot/asistente
        cursor.execute("""
        INSERT INTO INTERACCIONES_ASISTENTE (id_cliente, patron_consultado, insumos_sugeridos, conversion_exitosa, tiempo_sesion_segundos)
        VALUES (?, ?, ?, ?, ?)
        """, (cliente_id, patron_elegido, f"{hilado_rec} + {aguja_rec}", acepta_cross_selling, random.randint(30, 420)))

    conn.commit()
    conn.close()
    print(f"Simulación completa: {num_clientes} clientes y {num_compras} compras guardadas en 'tienda_crochet.db'.")

if __name__ == '__main__':
    generar_dataset_simulado()


Simulación completa: 150 clientes y 300 compras guardadas en 'tienda_crochet.db'.
